# Hybrid RAG with Re-Ranking — Self-Contained Demo

Everything needed to run the demo lives in **this notebook**.
No dependency on the `src/` folder — just the on-disk data and indexes.

### What you need locally
```
data/corpus/chunks.jsonl
indexes/dense/{embeddings.npy, faiss.index, dense_meta.json}
indexes/bm25/{bm25_model.pkl, tokenized_docs.pkl, bm25_meta.json}
```

### Python packages
```
pip install numpy faiss-cpu rank_bm25 sentence-transformers transformers torch
```

### Structure of this notebook
1. Imports & data classes
2. Load chunks, dense index, BM25 index
3. Dense retriever (inlined)
4. Sparse BM25 retriever (inlined)
5. Reciprocal Rank Fusion (inlined)
6. Cross-encoder re-ranker (inlined)
7. LLM generator + prompt builder (inlined)
8. End-to-end HybridRAG pipeline (inlined)
9. Demo — with/without reranker
10. Stage-by-stage inspection
11. Free-form query cell

## 1. Imports and data classes

In [ ]:
import json, os, pickle, time
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np


@dataclass(frozen=True)
class Chunk:
    chunk_id: str
    url: str
    title: str
    chunk_index: int
    text: str


@dataclass(frozen=True)
class RetrievedChunk:
    chunk: Chunk
    dense_score: float = 0.0
    bm25_score: float = 0.0
    rrf_score: float = 0.0
    dense_rank: Optional[int] = None
    bm25_rank: Optional[int] = None
    rerank_score: Optional[float] = None
    rerank_rank: Optional[int] = None


@dataclass(frozen=True)
class RAGAnswer:
    query: str
    answer: str
    context_chunks: List[RetrievedChunk]
    latency_ms: Dict[str, float]
    debug: Dict[str, Any] = field(default_factory=dict)

print('Data classes ready')

## 2. Load chunks and set paths

In [ ]:
PROJECT_ROOT = Path.cwd()
CHUNKS_PATH  = PROJECT_ROOT / 'data'    / 'corpus' / 'chunks.jsonl'
DENSE_DIR    = PROJECT_ROOT / 'indexes' / 'dense'
BM25_DIR     = PROJECT_ROOT / 'indexes' / 'bm25'

def read_jsonl(path: Path) -> List[dict]:
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

chunks: List[Chunk] = [Chunk(**r) for r in read_jsonl(CHUNKS_PATH)]
print(f'Loaded {len(chunks):,} chunks')
print(f'Example title: {chunks[0].title!r}')

## 3. Dense retriever

Encode the query with **sentence-transformers/all-mpnet-base-v2** and match it against a FAISS cosine-similarity index over the pre-computed chunk embeddings.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

def _normalize(matrix: np.ndarray) -> np.ndarray:
    return matrix / (np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-12)


class DenseIndex:
    def __init__(self, model_name: str = 'all-mpnet-base-v2'):
        self.model_name = model_name
        self.encoder: Optional[SentenceTransformer] = None
        self.embeddings: Optional[np.ndarray] = None
        self.faiss_index = None

    @classmethod
    def load(cls, input_dir: Path) -> 'DenseIndex':
        with open(input_dir / 'dense_meta.json', 'r', encoding='utf-8') as f:
            meta = json.load(f)
        inst = cls(model_name=meta['model_name'])
        inst.encoder = SentenceTransformer(inst.model_name)
        inst.embeddings = np.load(input_dir / 'embeddings.npy')
        faiss_path = input_dir / 'faiss.index'
        if faiss_path.exists():
            inst.faiss_index = faiss.read_index(str(faiss_path))
        return inst

    def search(self, query: str, top_k: int = 50) -> Tuple[np.ndarray, np.ndarray]:
        q = self.encoder.encode([query])
        q = _normalize(np.asarray(q, dtype=np.float32))
        if self.faiss_index is not None:
            scores, idx = self.faiss_index.search(q, top_k)
            return idx[0], scores[0]
        scores = np.dot(self.embeddings, q[0])
        ranked = np.argsort(-scores)[:top_k]
        return ranked, scores[ranked]


dense_index = DenseIndex.load(DENSE_DIR)
print('Dense index loaded -', dense_index.embeddings.shape)

## 4. Sparse retriever — BM25

Classic term-frequency / IDF ranking via `rank_bm25`. The pickled model in `indexes/bm25/` was trained on tokenized chunks.

In [ ]:
from rank_bm25 import BM25Okapi

def simple_tokenize(text: str) -> List[str]:
    return [t for t in text.lower().split() if t]


class BM25Index:
    def __init__(self):
        self.model: Optional[BM25Okapi] = None
        self.tokenized_documents: List[List[str]] = []

    @classmethod
    def load(cls, input_dir: Path) -> 'BM25Index':
        inst = cls()
        with open(input_dir / 'bm25_model.pkl', 'rb') as f:
            inst.model = pickle.load(f)
        with open(input_dir / 'tokenized_docs.pkl', 'rb') as f:
            inst.tokenized_documents = pickle.load(f)
        return inst

    def search(self, query: str, top_k: int = 50) -> Tuple[np.ndarray, np.ndarray]:
        tokens = simple_tokenize(query)
        scores = np.asarray(self.model.get_scores(tokens), dtype=np.float32)
        ranked = np.argsort(-scores)[:top_k]
        return ranked, scores[ranked]


bm25_index = BM25Index.load(BM25_DIR)
print('BM25 index loaded -', len(bm25_index.tokenized_documents), 'documents')

## 5. Reciprocal Rank Fusion

Combines the dense and sparse ranking lists. No score calibration required — it only uses the **rank position**.

$$ \text{RRF}(d) = \sum_i \frac{1}{k + \text{rank}_i(d)}, \quad k = 60 $$

In [ ]:
def reciprocal_rank_fusion(dense_idx: np.ndarray,
                           sparse_idx: np.ndarray,
                           fusion_constant: int = 60,
                           limit: Optional[int] = None) -> Dict[int, float]:
    if limit is not None:
        dense_idx  = dense_idx[:limit]
        sparse_idx = sparse_idx[:limit]
    fused: Dict[int, float] = {}
    for rank, doc in enumerate(dense_idx, start=1):
        fused[int(doc)] = fused.get(int(doc), 0.0) + 1.0 / (fusion_constant + rank)
    for rank, doc in enumerate(sparse_idx, start=1):
        fused[int(doc)] = fused.get(int(doc), 0.0) + 1.0 / (fusion_constant + rank)
    return fused


def top_n_by_score(score_map: Dict[int, float], top_n: int) -> List[Tuple[int, float]]:
    return sorted(score_map.items(), key=lambda kv: kv[1], reverse=True)[:top_n]

## 6. Cross-encoder re-ranker

The reranker scores each `(query, chunk)` pair **jointly** instead of separately — slower per-pair but much more accurate. We run it only on the top 25 fused candidates.

**First call downloads ~80 MB.**

In [ ]:
from sentence_transformers import CrossEncoder

class CrossEncoderReranker:
    def __init__(self, model_name: str = 'cross-encoder/ms-marco-MiniLM-L-6-v2',
                 device: str = 'cpu'):
        self.model_name = model_name
        self.device = device
        self._model: Optional[CrossEncoder] = None

    def load(self) -> None:
        if self._model is None:
            self._model = CrossEncoder(self.model_name, device=self.device)

    def rerank(self, query: str, candidates: List[RetrievedChunk],
               top_n: int) -> List[RetrievedChunk]:
        if not candidates:
            return []
        self.load()
        pairs  = [[query, c.chunk.text] for c in candidates]
        scores = self._model.predict(pairs)
        ranked = sorted(zip(candidates, [float(s) for s in scores]),
                        key=lambda x: x[1], reverse=True)
        return [replace(c, rerank_score=s, rerank_rank=i + 1)
                for i, (c, s) in enumerate(ranked[:top_n])]


reranker = CrossEncoderReranker()
reranker.load()
print('Reranker loaded')

## 7. LLM generator + prompt builder

`google/flan-t5-base` as the generator — small, CPU-friendly, instruction-tuned.

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

class LLMGenerator:
    def __init__(self, model_name: str = 'google/flan-t5-base', device: str = 'cpu'):
        self.model_name = model_name
        self.device = device
        self.model = None
        self.tokenizer = None

    def load(self) -> None:
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)
        self.model.to(self.device)

    def generate(self, prompt: str, max_new_tokens: int = 128) -> str:
        inputs = self.tokenizer(prompt, return_tensors='pt', truncation=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            out = self.model.generate(**inputs,
                                      max_new_tokens=max_new_tokens,
                                      do_sample=False)
        return self.tokenizer.decode(out[0], skip_special_tokens=True)


def build_prompt(query: str, context_chunks: List[RetrievedChunk]) -> str:
    blocks = []
    for i, rc in enumerate(context_chunks, start=1):
        blocks.append(f'Source {i}: {rc.chunk.title} ({rc.chunk.url})\n{rc.chunk.text}')
    return (
        'You are a helpful assistant. Answer the question using only the provided context. '
        'If the question has multiple parts, address every part in one complete sentence. '
        'Include all relevant facts such as locations, dates, years, and numbers. '
        'If the answer is not present in the context, say you do not know.\n\n'
        f'Question: {query}\n\n'
        f'Context:\n' + '\n\n'.join(blocks) + '\n\n'
        'Complete answer:'
    )


generator = LLMGenerator()
generator.load()
print('Flan-T5-base loaded')

## 8. End-to-end HybridRAG pipeline

Dense + BM25 → RRF → (optional cross-encoder rerank) → Flan-T5.

In [ ]:
class HybridRAG:
    def __init__(self, chunks, dense_index, bm25_index, generator,
                 rrf_constant: int = 60,
                 reranker: Optional[CrossEncoderReranker] = None):
        self.chunks = chunks
        self.dense_index = dense_index
        self.bm25_index = bm25_index
        self.generator = generator
        self.rrf_constant = rrf_constant
        self.reranker = reranker

    def retrieve(self, query: str, top_k: int = 50, context_size: int = 6,
                 use_reranker: bool = False,
                 rerank_pool: int = 25) -> List[RetrievedChunk]:
        dense_idx, dense_sc = self.dense_index.search(query, top_k)
        sparse_idx, sparse_sc = self.bm25_index.search(query, top_k)

        fused = reciprocal_rank_fusion(dense_idx, sparse_idx,
                                       fusion_constant=self.rrf_constant,
                                       limit=top_k)

        rerank_active = use_reranker and self.reranker is not None
        first_n = max(rerank_pool, context_size) if rerank_active else context_size
        fused_top = top_n_by_score(fused, first_n)

        d_rank = {int(d): i + 1 for i, d in enumerate(dense_idx)}
        s_rank = {int(d): i + 1 for i, d in enumerate(sparse_idx)}
        d_sc   = {int(d): float(s) for d, s in zip(dense_idx, dense_sc)}
        s_sc   = {int(d): float(s) for d, s in zip(sparse_idx, sparse_sc)}

        retrieved: List[RetrievedChunk] = []
        for doc_idx, rrf_s in fused_top:
            retrieved.append(RetrievedChunk(
                chunk=self.chunks[int(doc_idx)],
                dense_score=d_sc.get(int(doc_idx), 0.0),
                bm25_score=s_sc.get(int(doc_idx), 0.0),
                rrf_score=float(rrf_s),
                dense_rank=d_rank.get(int(doc_idx)),
                bm25_rank=s_rank.get(int(doc_idx)),
            ))

        if rerank_active:
            retrieved = self.reranker.rerank(query, retrieved, top_n=context_size)
        return retrieved

    def answer(self, query: str, top_k: int = 50, context_size: int = 6,
               max_new_tokens: int = 160,
               use_reranker: bool = False,
               rerank_pool: int = 25) -> RAGAnswer:
        t0 = time.perf_counter()
        tr0 = time.perf_counter()
        ctx = self.retrieve(query, top_k, context_size,
                            use_reranker=use_reranker, rerank_pool=rerank_pool)
        tr1 = time.perf_counter()
        prompt = build_prompt(query, ctx)
        tg0 = time.perf_counter()
        out  = self.generator.generate(prompt, max_new_tokens=max_new_tokens)
        tg1 = time.perf_counter()
        t1   = time.perf_counter()
        return RAGAnswer(
            query=query, answer=out, context_chunks=ctx,
            latency_ms={
                'retrieve_total': (tr1 - tr0) * 1000.0,
                'generate':       (tg1 - tg0) * 1000.0,
                'total':          (t1 - t0) * 1000.0,
            },
        )


rag = HybridRAG(chunks=chunks,
                dense_index=dense_index,
                bm25_index=bm25_index,
                generator=generator,
                reranker=reranker)
print('HybridRAG ready')

## 9. Pretty-printer

In [ ]:
def show(result: RAGAnswer, max_sources: int = 5, snippet_chars: int = 220):
    print('=' * 80)
    print(f'Q: {result.query}')
    print('-' * 80)
    print(f'A: {result.answer}')
    print('-' * 80)
    lat = result.latency_ms
    print(f"Latency  retrieve={lat['retrieve_total']:.0f}ms  "
          f"generate={lat['generate']:.0f}ms  total={lat['total']:.0f}ms")
    print('-' * 80)
    print('Top sources:')
    for i, c in enumerate(result.context_chunks[:max_sources], start=1):
        rr = f"  rerank={c.rerank_score:.3f}" if c.rerank_score is not None else ''
        print(f'  {i}. {c.chunk.title}')
        print(f'     dense_rank={c.dense_rank}  bm25_rank={c.bm25_rank}  '
              f'rrf={c.rrf_score:.4f}{rr}')
        snippet = c.chunk.text[:snippet_chars].replace('\n', ' ')
        print(f'     {snippet}...')
    print('=' * 80)
    print()

## 10. Demo — without and with reranker

Same query, same context size, only the reranker toggled.

In [ ]:
query = 'When was the Eiffel Tower built?'

print('>>> WITHOUT reranker')
show(rag.answer(query, use_reranker=False))

print('>>> WITH reranker')
show(rag.answer(query, use_reranker=True, rerank_pool=25))

## 11. Five demo queries, side by side

In [ ]:
demo_queries = [
    'When was the Eiffel Tower built?',
    'What is the height of Burj Khalifa?',
    'Which country is Machu Picchu located in?',
    'What is the Great Barrier Reef made of?',
    'Who designed the Sagrada Familia?',
]

for q in demo_queries:
    print('\n>>> WITHOUT reranker')
    show(rag.answer(q, use_reranker=False),
         max_sources=3, snippet_chars=120)
    print('>>> WITH reranker')
    show(rag.answer(q, use_reranker=True, rerank_pool=25),
         max_sources=3, snippet_chars=120)

## 12. Compound query — the reranker's moment

In [ ]:
compound = 'Where is the Burj Khalifa and when was it built?'

print('>>> WITHOUT reranker')
show(rag.answer(compound, use_reranker=False))

print('>>> WITH reranker')
show(rag.answer(compound, use_reranker=True, rerank_pool=25))

## 13. Stage-by-stage inspection

Raw dense top-10, raw BM25 top-10, and the fused RRF top-10.

In [ ]:
q = 'What is the height of Burj Khalifa?'

dense_idx, dense_sc = dense_index.search(q, top_k=10)
bm25_idx,  bm25_sc  = bm25_index.search(q,  top_k=10)

print('Top 10 - Dense')
for i, (d, s) in enumerate(zip(dense_idx, dense_sc), 1):
    print(f'  {i:2d}. {chunks[int(d)].title:<40}  score={s:.3f}')

print('\nTop 10 - BM25')
for i, (d, s) in enumerate(zip(bm25_idx, bm25_sc), 1):
    print(f'  {i:2d}. {chunks[int(d)].title:<40}  score={s:.3f}')

fused = reciprocal_rank_fusion(dense_idx, bm25_idx, fusion_constant=60)
print('\nTop 10 - Fused (RRF)')
for i, (d, s) in enumerate(top_n_by_score(fused, 10), 1):
    print(f'  {i:2d}. {chunks[int(d)].title:<40}  rrf={s:.4f}')

## 14. Tune rerank pool on the fly

In [ ]:
q = 'Who designed the Sagrada Familia?'

for pool in [10, 25, 40]:
    print(f'\n--- rerank_pool = {pool} ---')
    show(rag.answer(q, context_size=4, use_reranker=True, rerank_pool=pool),
         max_sources=4, snippet_chars=100)

## 15. Free-form query — audience suggestions

In [ ]:
my_query = 'What is the Colosseum in Rome known for?'
show(rag.answer(my_query, use_reranker=True, rerank_pool=25))

---
**Done.** All logic is inlined in this notebook — nothing is imported from `src/`.